# Tier 2 Project Parts

Space Elevator Phase 1 & 2 production planning.

All quantities in **items/min**. Blueprints from `blueprints.py`.

Each section defines:
1. **Supply** — raw input rate(s)
2. **Blueprint placement** — copies of each blueprint and the output rate you set in-game
3. **Assertions** — verify the chain is balanced (every intermediate rate matches)

In [1]:
from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS
import pandas as pd

BP = BLUEPRINTS  # shorthand used throughout

# Tolerance for balance assertions — accounts for rounding output rates to 2 d.p.
TOL = 0.5  # items/min


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    """Format per-machine rates for all internal stages as a hover string."""
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(
            f"  {r['machines']}× {r['building'].title()}"
            f" → {r['per_machine_rate']:.2f} {name}/min each"
        )
    return "<br>" + "<br>".join(lines)

---
## Smart Plating

**Supply:** 270 Iron Ingots/min (1 production line)  
**Target:** Maximise Smart Plating output

**Chain:**  
Iron Ingots → Iron Rods + Screws + Iron Plates → Rotors + Reinforced Iron Plates → Smart Plating

In [2]:
# ── Supply ────────────────────────────────────────────────────────────────────
IRON_INGOTS = 270.0  # items/min

# ── Blueprint placement ───────────────────────────────────────────────────────
# (copies, output rate per copy) — the two numbers you set in-game

ROD_COPIES,             ROD_OUTPUT_EACH             = 1, 58.06   # iron-rod
SCREW_COPIES,           SCREW_OUTPUT_EACH           = 3, 143.23  # screw
PLATE_COPIES,           PLATE_OUTPUT_EACH           = 1, 69.68   # iron-plate
REINFORCED_PLATE_COPIES, REINFORCED_PLATE_OUTPUT_EACH = 1, 11.61 # reinforced-iron-plate
ROTOR_COPIES,           ROTOR_OUTPUT_EACH           = 1, 11.61   # rotor
SMART_PLATING_COPIES,   SMART_PLATING_OUTPUT_EACH   = 2,  5.81   # smart-plating

### Derived rates and balance assertions

In [3]:
# ── Total production at each stage ────────────────────────────────────────────
rod_total             = ROD_COPIES             * ROD_OUTPUT_EACH
screw_total           = SCREW_COPIES           * SCREW_OUTPUT_EACH
plate_total           = PLATE_COPIES           * PLATE_OUTPUT_EACH
reinforced_plate_total = REINFORCED_PLATE_COPIES * REINFORCED_PLATE_OUTPUT_EACH
rotor_total           = ROTOR_COPIES           * ROTOR_OUTPUT_EACH
smart_plating_total   = SMART_PLATING_COPIES   * SMART_PLATING_OUTPUT_EACH

# ── Ingot consumption at each raw-input stage ─────────────────────────────────
# input_ratio(in, out) = how many units of 'in' per unit of 'out' at any clock
rod_ingots   = rod_total   * BP['iron-rod'].input_ratio('iron-ingot', 'iron-rod')
screw_ingots = screw_total * BP['screw'].input_ratio('iron-ingot', 'screw')
plate_ingots = plate_total * BP['iron-plate'].input_ratio('iron-ingot', 'iron-plate')
ingots_consumed = rod_ingots + screw_ingots + plate_ingots

# ── Intermediate consumption ──────────────────────────────────────────────────
plates_for_reinforced   = reinforced_plate_total * BP['reinforced-iron-plate'].input_ratio('iron-plate', 'reinforced-iron-plate')
screws_for_reinforced   = reinforced_plate_total * BP['reinforced-iron-plate'].input_ratio('screw',      'reinforced-iron-plate')
rods_for_rotor          = rotor_total            * BP['rotor'].input_ratio('iron-rod', 'rotor')
screws_for_rotor        = rotor_total            * BP['rotor'].input_ratio('screw',    'rotor')
reinforced_for_plating  = smart_plating_total    * BP['smart-plating'].input_ratio('reinforced-iron-plate', 'smart-plating')
rotors_for_plating      = smart_plating_total    * BP['smart-plating'].input_ratio('rotor',                 'smart-plating')

In [4]:
# ── Balance assertions ────────────────────────────────────────────────────────
# Each checks one junction. Failure message shows what's off and by how much.

assert abs(ingots_consumed - IRON_INGOTS) < TOL, (
    f"Iron Ingot balance: consuming {ingots_consumed:.2f}/min, supplied {IRON_INGOTS:.0f}/min "
    f"(delta {ingots_consumed - IRON_INGOTS:+.2f})"
)
assert abs(plate_total - plates_for_reinforced) < TOL, (
    f"Iron Plate balance: producing {plate_total:.2f}/min, "
    f"Reinforced Iron Plate needs {plates_for_reinforced:.2f}/min "
    f"(delta {plate_total - plates_for_reinforced:+.2f})"
)
assert abs(rod_total - rods_for_rotor) < TOL, (
    f"Iron Rod balance: producing {rod_total:.2f}/min, Rotor needs {rods_for_rotor:.2f}/min "
    f"(delta {rod_total - rods_for_rotor:+.2f})"
)
assert abs(screw_total - (screws_for_reinforced + screws_for_rotor)) < TOL, (
    f"Screw balance: producing {screw_total:.2f}/min, "
    f"needed {screws_for_reinforced + screws_for_rotor:.2f}/min "
    f"(delta {screw_total - screws_for_reinforced - screws_for_rotor:+.2f})"
)
assert abs(reinforced_plate_total - reinforced_for_plating) < TOL, (
    f"Reinforced Iron Plate balance: producing {reinforced_plate_total:.2f}/min, "
    f"Smart Plating needs {reinforced_for_plating:.2f}/min "
    f"(delta {reinforced_plate_total - reinforced_for_plating:+.2f})"
)
assert abs(rotor_total - rotors_for_plating) < TOL, (
    f"Rotor balance: producing {rotor_total:.2f}/min, Smart Plating needs {rotors_for_plating:.2f}/min "
    f"(delta {rotor_total - rotors_for_plating:+.2f})"
)

print(f"✓ Chain balanced")
print(f"  {IRON_INGOTS:.0f} Iron Ingots/min  →  {smart_plating_total:.2f} Smart Plating/min")

✓ Chain balanced
  270 Iron Ingots/min  →  11.62 Smart Plating/min


In [5]:
# ── Placement summary ─────────────────────────────────────────────────────────
rows = [
    ('Iron Rod',              ROD_COPIES,              f'{ROD_OUTPUT_EACH:.2f}/min',              f'{rod_total:.2f}/min',              f'{rod_ingots:.2f}',   '—'),
    ('Screw',                 SCREW_COPIES,            f'{SCREW_OUTPUT_EACH:.2f}/min',            f'{screw_total:.2f}/min',            f'{screw_ingots:.2f}', '—'),
    ('Iron Plate',            PLATE_COPIES,            f'{PLATE_OUTPUT_EACH:.2f}/min',            f'{plate_total:.2f}/min',            f'{plate_ingots:.2f}', '—'),
    ('Reinforced Iron Plate', REINFORCED_PLATE_COPIES, f'{REINFORCED_PLATE_OUTPUT_EACH:.2f}/min', f'{reinforced_plate_total:.2f}/min', '—',                  f'{plates_for_reinforced:.2f} Iron Plates + {screws_for_reinforced:.2f} Screws'),
    ('Rotor',                 ROTOR_COPIES,            f'{ROTOR_OUTPUT_EACH:.2f}/min',            f'{rotor_total:.2f}/min',            '—',                  f'{rods_for_rotor:.2f} Iron Rods + {screws_for_rotor:.2f} Screws'),
    ('Smart Plating',         SMART_PLATING_COPIES,    f'{SMART_PLATING_OUTPUT_EACH:.2f}/min',    f'{smart_plating_total:.2f}/min',    '—',                  f'{reinforced_for_plating:.2f} Reinforced Iron Plates + {rotors_for_plating:.2f} Rotors'),
]

pd.DataFrame(
    rows,
    columns=['Blueprint', 'Copies', 'Set each to', 'Total out', 'Ingots/min', 'Consumes'],
).set_index('Blueprint')

,Copies,Set each to,Total out,Ingots/min,Consumes
Blueprint,,,,,
Iron Rod,1,58.06/min,58.06/min,58.06,—
Screw,3,143.23/min,429.69/min,107.42,—
Iron Plate,1,69.68/min,69.68/min,104.52,—
Reinforced Iron Plate,1,11.61/min,11.61/min,—,69.66 Iron Plates + 139.32 Screws
Rotor,1,11.61/min,11.61/min,—,58.05 Iron Rods + 290.25 Screws
Smart Plating,2,5.81/min,11.62/min,—,11.62 Reinforced Iron Plates + 11.62 Rotors


### Production flow

Arrow width is proportional to **ingot-equivalent cost** — how many Iron Ingots each flow represents.  
This keeps every node perfectly balanced (in = out) and makes the chart readable across all stages.  
Hover over any node or arrow for actual items/min values.

In [6]:
import plotly.graph_objects as go

# ── Ingot-equivalent cost per item ────────────────────────────────────────────
ingot_per_rod         = BP['iron-rod'].input_ratio('iron-ingot', 'iron-rod')
ingot_per_screw       = BP['screw'].input_ratio('iron-ingot', 'screw')
ingot_per_plate       = BP['iron-plate'].input_ratio('iron-ingot', 'iron-plate')
ingot_per_reinforced  = (
    BP['reinforced-iron-plate'].input_ratio('iron-plate', 'reinforced-iron-plate') * ingot_per_plate +
    BP['reinforced-iron-plate'].input_ratio('screw',      'reinforced-iron-plate') * ingot_per_screw
)
ingot_per_rotor       = (
    BP['rotor'].input_ratio('iron-rod', 'rotor') * ingot_per_rod +
    BP['rotor'].input_ratio('screw',    'rotor') * ingot_per_screw
)
ingot_per_smart_plating = (
    BP['smart-plating'].input_ratio('reinforced-iron-plate', 'smart-plating') * ingot_per_reinforced +
    BP['smart-plating'].input_ratio('rotor',                 'smart-plating') * ingot_per_rotor
)

# ── Colour palette ────────────────────────────────────────────────────────────
SUPPLY_COL = "#475569"
TIER1_COL  = "#1d4ed8"
TIER2_COL  = "#0f766e"
FINAL_COL  = "#b45309"
LINK_COL   = "rgba(148, 163, 184, 0.25)"

# ── Nodes — labels show per-copy setting, hover shows per-machine rates ────────
nodes = [
    (f"Iron Ingots<br>{IRON_INGOTS:.0f}/min",
     SUPPLY_COL,
     f"Supply: {IRON_INGOTS:.0f} Iron Ingots/min"),

    (f"Iron Rod ×{ROD_COPIES}<br>set each to {ROD_OUTPUT_EACH:.2f}/min",
     TIER1_COL,
     f"{ROD_COPIES} blueprint{'s' if ROD_COPIES>1 else ''}<br>"
     f"Set each output to: {ROD_OUTPUT_EACH:.2f} Iron Rods/min<br>"
     f"Total: {rod_total:.2f} Iron Rods/min"
     + machine_hover(BP['iron-rod'], 'iron-rod', ROD_OUTPUT_EACH)),

    (f"Screw ×{SCREW_COPIES}<br>set each to {SCREW_OUTPUT_EACH:.2f}/min",
     TIER1_COL,
     f"{SCREW_COPIES} blueprints<br>"
     f"Set each output to: {SCREW_OUTPUT_EACH:.2f} Screws/min<br>"
     f"Total: {screw_total:.2f} Screws/min"
     + machine_hover(BP['screw'], 'screw', SCREW_OUTPUT_EACH)),

    (f"Iron Plate ×{PLATE_COPIES}<br>set each to {PLATE_OUTPUT_EACH:.2f}/min",
     TIER1_COL,
     f"{PLATE_COPIES} blueprint{'s' if PLATE_COPIES>1 else ''}<br>"
     f"Set each output to: {PLATE_OUTPUT_EACH:.2f} Iron Plates/min<br>"
     f"Total: {plate_total:.2f} Iron Plates/min"
     + machine_hover(BP['iron-plate'], 'iron-plate', PLATE_OUTPUT_EACH)),

    (f"Rotor ×{ROTOR_COPIES}<br>set each to {ROTOR_OUTPUT_EACH:.2f}/min",
     TIER2_COL,
     f"{ROTOR_COPIES} blueprint{'s' if ROTOR_COPIES>1 else ''}<br>"
     f"Set each output to: {ROTOR_OUTPUT_EACH:.2f} Rotors/min<br>"
     f"Consumes per copy: {rods_for_rotor/ROTOR_COPIES:.2f} Iron Rods/min"
     f" + {screws_for_rotor/ROTOR_COPIES:.2f} Screws/min"
     + machine_hover(BP['rotor'], 'rotor', ROTOR_OUTPUT_EACH)),

    (f"Reinforced Iron Plate ×{REINFORCED_PLATE_COPIES}<br>set each to {REINFORCED_PLATE_OUTPUT_EACH:.2f}/min",
     TIER2_COL,
     f"{REINFORCED_PLATE_COPIES} blueprint{'s' if REINFORCED_PLATE_COPIES>1 else ''}<br>"
     f"Set each output to: {REINFORCED_PLATE_OUTPUT_EACH:.2f} Reinforced Iron Plates/min<br>"
     f"Consumes per copy: {plates_for_reinforced/REINFORCED_PLATE_COPIES:.2f} Iron Plates/min"
     f" + {screws_for_reinforced/REINFORCED_PLATE_COPIES:.2f} Screws/min"
     + machine_hover(BP['reinforced-iron-plate'], 'reinforced-iron-plate', REINFORCED_PLATE_OUTPUT_EACH)),

    (f"Smart Plating ×{SMART_PLATING_COPIES}<br>set each to {SMART_PLATING_OUTPUT_EACH:.2f}/min",
     FINAL_COL,
     f"{SMART_PLATING_COPIES} blueprints<br>"
     f"Set each output to: {SMART_PLATING_OUTPUT_EACH:.2f} Smart Plating/min<br>"
     f"Consumes per copy: {reinforced_for_plating/SMART_PLATING_COPIES:.2f} Reinforced Iron Plates/min"
     f" + {rotors_for_plating/SMART_PLATING_COPIES:.2f} Rotors/min"
     + machine_hover(BP['smart-plating'], 'smart-plating', SMART_PLATING_OUTPUT_EACH)),
]

# ── Links: ingot-equivalent values, hover shows actual items/min ───────────────
links = [
    (0, 1, rod_total    * ingot_per_rod,
     f"{rod_ingots:.2f} Iron Ingots/min → Iron Rod blueprints<br>({rod_total:.2f} Iron Rods/min)"),
    (0, 2, screw_total  * ingot_per_screw,
     f"{screw_ingots:.2f} Iron Ingots/min → Screw blueprints<br>({screw_total:.2f} Screws/min)"),
    (0, 3, plate_total  * ingot_per_plate,
     f"{plate_ingots:.2f} Iron Ingots/min → Iron Plate blueprints<br>({plate_total:.2f} Iron Plates/min)"),
    (1, 4, rod_total             * ingot_per_rod,   f"{rod_total:.2f} Iron Rods/min → Rotor blueprints"),
    (2, 4, screws_for_rotor      * ingot_per_screw, f"{screws_for_rotor:.2f} Screws/min → Rotor blueprints"),
    (2, 5, screws_for_reinforced * ingot_per_screw, f"{screws_for_reinforced:.2f} Screws/min → Reinforced Iron Plate blueprints"),
    (3, 5, plates_for_reinforced * ingot_per_plate, f"{plates_for_reinforced:.2f} Iron Plates/min → Reinforced Iron Plate blueprints"),
    (4, 6, rotor_total            * ingot_per_rotor,        f"{rotor_total:.2f} Rotors/min → Smart Plating blueprints"),
    (5, 6, reinforced_plate_total * ingot_per_reinforced,   f"{reinforced_plate_total:.2f} Reinforced Iron Plates/min → Smart Plating blueprints"),
]

# ── Figure ────────────────────────────────────────────────────────────────────
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=20,
        thickness=24,
        line=dict(color="#0f172a", width=1.5),
        label=[n[0] for n in nodes],
        color=[n[1] for n in nodes],
        customdata=[n[2] for n in nodes],
        hovertemplate="%{customdata}<extra></extra>",
    ),
    link=dict(
        source=[l[0] for l in links],
        target=[l[1] for l in links],
        value= [l[2] for l in links],
        customdata=[l[3] for l in links],
        color=LINK_COL,
        hovertemplate="%{customdata}<extra></extra>",
    ),
))

fig.update_layout(
    title=dict(
        text=(f"Smart Plating — "
              f"{IRON_INGOTS:.0f} Iron Ingots/min → {smart_plating_total:.2f} Smart Plating/min"),
        font=dict(color="#e2e8f0", size=15),
        x=0.01,
    ),
    paper_bgcolor="#0f172a",
    font=dict(color="#e2e8f0", size=11, family="monospace"),
    height=500,
    margin=dict(l=20, r=20, t=50, b=20),
)

fig.show()